# Data Cleaning — Left-Wing Non-Fiction (Scraping)
Loads `Data/Raw/non fiction/leftpolitics_raw(scraping).csv`, cleans it, and saves to `Data/non fiction/leftpolitics_clean(scraping).csv`.

In [3]:
import pandas as pd
import os

In [4]:
RAW_PATH = "../Data/Raw/non fiction/leftpolitics_raw(scraping).csv"
CLEAN_DIR = "../Data/non fiction"
CLEAN_PATH = os.path.join(CLEAN_DIR, "leftpolitics_clean(scraping).csv")

df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()

Loaded 5187 rows, 7 columns


,title,author,year_published,subjects,edition_count,open_library_key,queried_author
0,La naissance du maquis dans le Sud-Cameroun (H...,Achille Mbembe,1996.0,NaN,2,/book/show/74573817-la-naissance-du-maquis-dan...,Achille Mbembe
1,On the Postcolony,Achille Mbembe,2001.0,NaN,17,/book/show/149757.On_the_Postcolony,Achille Mbembe
2,Johannesburg: The Elusive Metropolis (Volume 16),Sarah Nuttall,2004.0,NaN,5,/book/show/327816.Johannesburg,Achille Mbembe
3,"África Insubmissa: Cristianismo, poder e estad...",Achille Mbembe,2005.0,NaN,2,/book/show/45914660-frica-insubmissa,Achille Mbembe
4,Je est un autre - Pour une identité-monde,Kebir-Mustapha Ammi,2010.0,NaN,3,/book/show/44436174-je-est-un-autre---pour-une...,Achille Mbembe


In [5]:
print("=== Before cleaning ===")
print(df.dtypes)
print()
print(df.isnull().sum())

=== Before cleaning ===
title                object
author               object
year_published      float64
subjects            float64
edition_count         int64
open_library_key     object
queried_author       object
dtype: object

title                  0
author                 1
year_published      3642
subjects            5187
edition_count          0
open_library_key       0
queried_author         0
dtype: int64


In [6]:
df_clean = df.copy()

# --- Strip whitespace from all string columns ---
str_cols = df_clean.select_dtypes(include="object").columns
df_clean[str_cols] = df_clean[str_cols].apply(lambda col: col.str.strip())

# --- Fill missing author from queried_author ---
mask = df_clean["author"].isnull()
df_clean.loc[mask, "author"] = df_clean.loc[mask, "queried_author"]
print(f"Filled {mask.sum()} missing author(s) from queried_author")

# --- Drop rows still missing title or author ---
before = len(df_clean)
df_clean = df_clean.dropna(subset=["title", "author"])
df_clean = df_clean[df_clean["title"].str.len() > 0]
df_clean = df_clean[df_clean["author"].str.len() > 0]
print(f"Dropped {before - len(df_clean)} rows missing title or author")

# --- Standardize author names: title-case, collapse extra spaces ---
def standardize_name(name):
    return " ".join(name.title().split())

df_clean["author"] = df_clean["author"].apply(standardize_name)
df_clean["queried_author"] = df_clean["queried_author"].apply(standardize_name)

# --- Make year numeric ---
df_clean["year_published"] = pd.to_numeric(df_clean["year_published"], errors="coerce")

# --- subjects column is empty from scraping — fill with empty string ---
df_clean["subjects"] = df_clean["subjects"].fillna("")

# --- Rename open_library_key to goodreads_path (it contains Goodreads links) ---
df_clean = df_clean.rename(columns={"open_library_key": "goodreads_path"})

# --- Drop exact duplicates (title + author) ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title", "author"])
print(f"Dropped {before - len(df_clean)} duplicate rows")

# --- Reset index ---
df_clean = df_clean.reset_index(drop=True)

print(f"\nClean dataset: {len(df_clean)} rows")
df_clean.head(10)

Filled 1 missing author(s) from queried_author
Dropped 0 rows missing title or author
Dropped 3 duplicate rows

Clean dataset: 5184 rows


,title,author,year_published,subjects,edition_count,goodreads_path,queried_author
0,La naissance du maquis dans le Sud-Cameroun (H...,Achille Mbembe,1996.0,,2,/book/show/74573817-la-naissance-du-maquis-dan...,Achille Mbembe
1,On the Postcolony,Achille Mbembe,2001.0,,17,/book/show/149757.On_the_Postcolony,Achille Mbembe
2,Johannesburg: The Elusive Metropolis (Volume 16),Sarah Nuttall,2004.0,,5,/book/show/327816.Johannesburg,Achille Mbembe
3,"África Insubmissa: Cristianismo, poder e estad...",Achille Mbembe,2005.0,,2,/book/show/45914660-frica-insubmissa,Achille Mbembe
4,Je est un autre - Pour une identité-monde,Kebir-Mustapha Ammi,2010.0,,3,/book/show/44436174-je-est-un-autre---pour-une...,Achille Mbembe
5,Necropolitics,Achille Mbembe,2016.0,,24,/book/show/44901216-necropolitics,Achille Mbembe
6,Politiques de l'inimitié,Achille Mbembe,2016.0,,2,/book/show/30246397-politiques-de-l-inimiti,Achille Mbembe
7,Brutalisme,Achille Mbembe,2020.0,,18,/book/show/51131710-brutalisme,Achille Mbembe
8,The Earthly Community: Reflections on the Last...,Achille Mbembe,2023.0,,15,/book/show/211097016-the-earthly-community,Achille Mbembe
9,Critique of Black Reason,Achille Mbembe,NaN,,4,/book/show/30757980-critique-of-black-reason,Achille Mbembe


In [7]:
print("=== After cleaning ===")
print(df_clean.dtypes)
print()
print(df_clean.isnull().sum())

=== After cleaning ===
title              object
author             object
year_published    float64
subjects           object
edition_count       int64
goodreads_path     object
queried_author     object
dtype: object

title                0
author               0
year_published    3639
subjects             0
edition_count        0
goodreads_path       0
queried_author       0
dtype: int64


In [8]:
os.makedirs(CLEAN_DIR, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(df_clean)} rows to {CLEAN_PATH}")

Saved 5184 rows to ../Data/non fiction/leftpolitics_clean(scraping).csv
